# Circulation

In [1]:
import xarray as xr
import dask
from dask_jobqueue import PBSCluster
from dask.distributed import Client

In [2]:
cluster = PBSCluster(
    cores=4, # The number of cores you want
    memory='64GB', # Amount of memory (resource_spec is the one)
    processes=1, # How many processes
    queue='casper', # The type of queue to utilize (/glade/u/apps/dav/opt/usr/bin/execcasper)
    local_directory='$TMPDIR', # Use your local directory
    account='P93300313', # Input your project ID here
    walltime='6:00:00', # Amount of wall time
)
cluster.scale(jobs=6)
# Setup your client
client = Client(cluster)
client

Connection method: Cluster object,Cluster type: dask_jobqueue.PBSCluster
Dashboard: http://128.117.208.213:8787/status,
Dashboard: http://128.117.208.213:8787/status,Workers: 0
Total threads: 0,Total memory: 0 B
Comm: tcp://128.117.208.213:35811,Workers: 0
Dashboard: http://128.117.208.213:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B


In [3]:
# chunks='auto' appproaches this value
dask.config.set({'array.chunk-size': '8 GiB'})

In [4]:
path_to_members = '/glade/campaign/cgd/ccr/E3SMv2/FV_regridded'
path_to_monthly = '/atm/proc/tseries/month_1/'


# member names
m_names = ['0101', '0111', '0121', '0131', '0141', '0151', '0161',
           '0171', '0181', '0191', '0201', '0211', '0221', '0231',
           '0241', '0251', '0261', '0271', '0281', '0291', '0301']

# member header
m_h = 'v2.FV1.historical_'

# variable
var = 'U'

# member file paths
m_fps = []
for n in m_names:
    file = m_h + n + '.eam.h0.' + var + '.185001-201412.nc'
    file_path = path_to_members + '/' + m_h + n + path_to_monthly + file
    m_fps.append(file_path)

In [5]:
def pre_slice(ds):
    # select var
    ds = ds['U']
    # sort lon
    ds.coords['lon'] = (ds.coords['lon'] + 180) % 360 - 180
    ds = ds.sortby(ds.lon)
    # sort lat
    ds.coords['lat'] = (ds.coords['lat'] + 90) % 180 - 90
    ds = ds.sortby(ds.lat)
    scoped_ds = ds.sel(lat=slice(-10, 10), lon=slice(-180, 20))
    return scoped_ds

In [6]:
Uds = xr.open_mfdataset(m_fps, combine='nested',
                          engine='h5netcdf', parallel=True,
                          concat_dim=['member'],
                          preprocess=pre_slice)
Uds.chunk(chunks='auto')

<xarray.DataArray 'OMEGA' (member: 21, time: 1980, lev: 72, lat: 22, lon: 161)> Size: 42GB
dask.array<rechunk-merge, shape=(21, 1980, 72, 22, 161), dtype=float32, chunksize=(21, 400, 72, 22, 161), chunktype=numpy.ndarray>
Coordinates:
  * time     (time) object 16kB 1850-02-01 00:00:00 ... 2015-01-01 00:00:00
  * lev      (lev) float64 576B 0.1238 0.1828 0.2699 ... 986.2 993.8 998.5
  * lat      (lat) float64 176B -9.895 -8.953 -8.01 -7.068 ... 8.01 8.953 9.895
  * lon      (lon) float64 1kB -180.0 -178.8 -177.5 -176.2 ... 17.5 18.75 20.0
Dimensions without coordinates: member
Attributes:
    mdims:          1
    units:          Pa/s
    long_name:      Vertical velocity (pressure)
    standard_name:  lagrangian_tendency_of_air_pressure
    cell_methods:   time: mean
    cell_measures:  area: area

In [7]:
Uds.to_zarr('/glade/work/acruz/E3SMv2LE/E3SMv2le_tropic_U', mode='w', consolidated=True)

/glade/u/home/acruz/.conda/envs/analysis_11_20_2025/lib/python3.11/site-packages/zarr/api/asynchronous.py:247: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


/glade/u/home/acruz/.conda/envs/analysis_11_20_2025/lib/python3.11/site-packages/distributed/client.py:3374: UserWarning: Sending large graph of size 18.00 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


In [8]:
client.shutdown()

In [9]:
quit()